# Lekcja 18 (kontynuacja): Pokwitowania, które dowodzą, że *człowiek* zatwierdził działanie

Lekcja pokazuje, co zrobił **agent** i co zdecydowała **brama**. Ten notatnik dodaje brakującą połowę: dowód, że **określony człowiek** zatwierdził **dokładne** działanie — osobny podpis człowieka nad pełnym kanonicznym działaniem, zweryfikowany offline.

Oba artefakty tutaj używają **takiego samego formatu koperty jak pokwitowania z lekcji**: płaski ładunek z polem `type`, podpisany Ed25519 bezpośrednio po kanonicznych bajtach JCS, z dołączonym (i wyłączonym z podpisania) strukturalnym obiektem `signature`. Pokwitowanie zatwierdzenia to nowy `type` (`human.approval.v1`) obok typu działania, więc jedna funkcja `verify_chain` obsługuje oba rodzaje artefaktów tym samym kodem, który zbudowałeś w głównym notatniku. To pokwitowanie zatwierdzenia przez człowieka to kompozycja edukacyjna zdefiniowana tutaj, a nie typ pokwitowania zdefiniowany przez draft-farley-acta-signed-receipts.

Jedna celowa ulepszona funkcja weryfikatora w porównaniu do tego z głównego notatnika: tutaj weryfikator odwołuje się do `signature.key_id` w **zarejestrowanym stałym kluczu** zamiast ufać kluczowi publicznemu zawartemu w pokwitowaniu. To jest produkcyjna postawa, którą zaleca lista kontrolna lekcji („opublikuj publiczny klucz do weryfikacji”), i to sprawia, że fałszerstwo skutkuje odmową zamiast obejściem typu „przynieś swój klucz”.

Zasada, której uczy ten notatnik: **podpisane zatwierdzenie samo w sobie nie jest upoważnieniem.** Upoważnienie istnieje tylko wtedy, gdy pokwitowanie zatwierdzenia i pokwitowanie działania nadal odnoszą się do tego samego kanonicznego działania w czasie wykonania, zgodnie z aktualną wersją polityki, kluczem i datą ważności, i gdy zatwierdzenie nie zostało już wykorzystane. Każda odmowa ma **unikalny powód**, więc możesz odróżnić *upoważnienie wygasło* od *wykonane działanie się zmieniło*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Dokładna akcja

Jednostką zatwierdzenia jest **kanoniczny obiekt akcji** — nie niejasna etykieta jak „zatwierdź zwrot”, lecz precyzyjna, w pełni określona akcja. Podpisanie całego obiektu (i wyprowadzenie z niego skrótu) pozwala nam później udowodnić, że człowiek zatwierdził *to* i nic więcej.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Jedna koperta, dwie władze

Każde potwierdzenie to koperta lekcji: płaski ładunek z polem `type` oraz obiektem `signature` (`alg`, `sig`, `key_id`), które **nie** są częścią podpisanych bajtów. `verify_envelope` to wspólna kontrola strukturalna + podpisu dla obu rodzajów potwierdzeń; to, przeciwko **rejestrowi przypiętych kluczy** którego rozstrzyga `signature.key_id`, oddziela władze:

- **potwierdzenie zatwierdzenia** (`human.approval.v1`) — nazwany zatwierdzający, pełna kanoniczna akcja **i jej skrót**, `policy_version`, sygnatury czasowe wystawienia i wygaśnięcia. Jednorazowe wykorzystanie jest śledzone na poziomie łańcucha.
- **potwierdzenie akcji** (`agent.action.v1`) — tożsamość agenta, `run_id`, ten sam kanoniczny skrót akcji, wynik wykonania + sygnatura czasowa oraz `parent_approval_ref`: `receipt_hash` zatwierdzenia, według tego samego konwencjonalnego schematu co `previous_receipt_hash` w łańcuchu lekcji.

Wspólne pole `action_digest` jest spoiwem, od którego zależy powiązanie. `key_id` znajduje się w obiekcie podpisu tylko jako wskazówka do wyszukiwania: skierowanie go na inny przypięty klucz powoduje niepowodzenie weryfikacji podpisu i nic nie wnosi.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: gdzie wiązanie jest faktycznie decydowane

`verify_chain` **nie jest** wygodnym wrapperem nad dwiema kontrolami podpisów. Jest to jedno miejsce, gdzie sprawdzane są razem wspólny kanoniczny `action_digest`, polityka/klucz/ważność zatwierdzenia oraz **jednorazowe zużycie** zatwierdzenia, względem akcji wykonywanej *w tej chwili*.

Każda odmowa ma **odrębny powód**, dzięki czemu czytelnik odmowy może stwierdzić, czy autoryzacja wygasła (polityka się zmieniła, klucz został obrócony, zatwierdzenie wygasło, zatwierdzenie zostało zużyte) lub czy wykonywana akcja zmieniła się pod jeszcze ważnym zatwierdzeniem (podstawienie digest).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Co wiązanie wychwytuje

Każdy z poniższych przypadków kończy się **zamknięciem** z **różnym powodem**. Pierwszy blok to klasyczny zestaw (manipulacja, zdekonfundowany pełnomocnik, powtórka, fałszerstwo na dowolnym uprawnieniu, błędne dane wejściowe). Drugi blok to para, która sprawia, że właściwość jest rzeczywista, a nie tylko stwierdzona:

- **przestarzałe uprawnienie** — podpis jest nadal ważny, ale wersja polityki została zmieniona, klucz zatwierdzającego został usunięty z przypiętego rejestru lub zatwierdzenie wygasło przed wykonaniem;
- **zamiana skrótu** — ważnie podpisany potwierdzenie działania, którego `parent_approval_ref` wskazuje na *prawdziwe* zatwierdzenie, ale kanoniczny skrót akcji tego zatwierdzenia nie zgadza się z faktycznie wykonywaną akcją.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Co to udowadnia — a czego nie  

**Udowadnia:** że nazwany człowiek zatwierdził *dokładnie tę kanoniczną akcję* (pełna akcja + skrót, podpisane kluczem rozwiązanym z przypiętego rejestru), a agent wykonał *dokładnie tę zatwierdzoną akcję* (ten sam skrót, kwit powiązany z zatwierdzeniem przez `receipt_hash`, własna konwencja łańcucha lekcji) — podczas gdy wersja polityki zatwierdzenia, klucz i data ważności były aktualne, dokładnie raz. Jeśli któraś ze stron się zmieni, łańcuch zamyka się na niepowodzeniu, a powód odmowy mówi ci, **które** właściwości zostały naruszone: przestarzały autorytet vs. zmieniona akcja.  

**Nie udowadnia:** że interfejs zatwierdzenia pokazał człowiekowi to, co myślał, że podpisuje (WYSIWYS to odrębny problem), że klucz nie został przymuszony lub skradziony przed rotacją, ani że efekty następcze odpowiadały akcji. Podpisane ≠ autoryzowane: ważny podpis nad przestarzałą polityką, obrócony klucz, wygasłe okno lub inny skrót nie daje tu niczego.  

Oba rodzaje kwitów świadomie współdzielą konwencję koperty lekcji i jedną ścieżkę kodu `verify_chain`: powiązanie, które zbudowałeś dla kwitów akcji w głównym notatniku, to ten sam kod sprawdzający zatwierdzenie człowieka. Jeden kontrakt weryfikatora, oddzielne przypięte autorytety, połączone kanonicznym skrótem akcji i niczym więcej.  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
